# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Research question:** Which observable content and query-demand signals are associated with a page achieving page-one search visibility, and can a simple learned model rank likely page-one pages better than a transparent single-signal baseline -- generalizing to clients it has never seen?

**The decision this supports:** which of thousands of content pages a content editor with limited review hours should look at first for page-one potential -- and which already-succeeding pages should be left alone rather than disturbed.

**The action:** an editor works down a ranked, reason-coded OPPORTUNITY queue first (expanding thin sections, pursuing backlinks, tightening focus), while PROTECT-tier pages (already page-one) are monitored rather than edited.

**Why this needs data/ML, not a hand-written rule:** the individual signals found here are each only moderately predictive on their own (10-14 percentage-point gaps) and, in one case, point in the *opposite* direction from common intuition (shorter content, not longer, associates with page-one). A hand-picked rule using the "obvious" signal in the "obvious" direction would already be wrong on the strongest individual signal -- which is exactly the situation a learned model, combining several moderate and sometimes counter-intuitive signals, is suited for.

In [1]:
print("Lane: Ranking Signal Analysis")
print("Question, decision, and action stated above -- reasoning only, no query needed.")


Lane: Ranking Signal Analysis
Question, decision, and action stated above -- reasoning only, no query needed.


## 2. Data

**Release:** `FlyRank/internship-warehouse` on Hugging Face (gated, instant-approval), build `v20260703`.

**Tables used:** `fact_content_daily_performance` (the daily fact, full partitioned table -- never the `_sample.parquet` file, which is the sealed final month of June 2026 and was deliberately avoided per the ML-04 card's warning), `dim_content` (content/keyword metadata), `dim_clients` (context only -- panel history checks and the client-grouping key for splits).

**Date window:** `month = '2026-03'` -- a mid-panel month, chosen specifically so the sealed June 2026 month stays untouched for any future genuinely-held-out test.

**Grain:** one row per `content_hash_id`, aggregated from the daily fact and joined to `dim_content` metadata.

**Label:** `is_page_one` = 1 if a page's impression-weighted average GSC position (`SUM(gsc_sum_position)/SUM(gsc_impressions)`) falls between 1 and 10 over the month, else 0. This is a **current-state, cross-sectional** label -- not a future prediction.

**What I excluded, and why:**
- Rows with `gsc_data_available = FALSE` -- these are zero-filled "instrument not yet on" rows, not real zero-traffic observations (confirmed directly in the raw data during ML-04).
- `gsc_avg_position`, `gsc_sum_position`, `gsc_clicks` as **features** -- the label and its direct derivative; confirmed as leaks via a deliberate confession test (ML-04, ML-09).
- `fact_content_query_90d` (query-mix) features -- tested explicitly (ML-05) and found to leak (+0.171 AUC gap when added), likely from its 90-day window overlapping the March label window. Excluded rather than assumed safe.
- Pages with `content_age_days < 90` and `gsc_impressions_win < 100` -- too young or too low-exposure for a fair position estimate.
- Unpublished/deleted pages (`is_published = FALSE` or `is_deleted = TRUE`).

In [2]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT    = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
MONTH   = "2026-03"

page_agg = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions_win,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()
page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]

content = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume, competition, backlinks, content_type, main_intent,
           GREATEST(date_diff('day', content_created_date, DATE '{MONTH}-01'), 0) AS content_age_days
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

data = con.sql("SELECT * FROM page_agg").df() if False else page_agg.merge(
    content, on="content_hash_id", how="left"
).dropna(subset=["word_count", "search_volume", "competition", "backlinks"]).reset_index(drop=True)

print(f"final data contract satisfied: {len(data):,} pages | {data['client_hash_id'].nunique()} clients | page-one rate {data['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

final data contract satisfied: 53,892 pages | 31 clients | page-one rate 56.2%


## 3. Methodology

**Features (5 numeric + 2 categorical, all confirmed leakage-safe):** `word_count`, `content_age_days`, `search_volume`, `competition`, `backlinks` (numeric); `content_type`, `main_intent` (categorical, one-hot). Median imputation for numeric, most-frequent for categorical.

**Baseline:** a transparent, hand-readable rule -- 4 binary conditions (`high_demand`, `low_competition`, `strong_backlinks`, `established`) summed 0-4, no fitted weights, with a continuous rank-based tiebreak (backlinks + search_volume + inverse-competition ranks averaged) to reduce noisy ties. Thresholds use **nonzero-only medians** after discovering `search_volume`/`competition`/`backlinks` are heavily zero-inflated (all-rows medians collapsed to 0.0 and made the rule nearly meaningless -- a real bug caught and fixed during ML-07).

**Model:** Logistic Regression (`class_weight='balanced'`), chosen first for its interpretability -- standardized coefficients read directly as signed, ranked signal strength. Random Forest trained alongside as a nonlinearity check.

**Validation design -- grouped by client, not random.** `GroupShuffleSplit(test_size=0.25, random_state=42)` on `client_hash_id`, so no client's pages appear in both train and test. **This choice was tested, not assumed:** re-running the same model under an ungrounded random row-level split showed AUC 0.654 / AP 0.656 versus the honest grouped split's AUC 0.561 / AP 0.589 (ML-09) -- a real +0.093 AUC / +0.068 AP illusion from client-memorization that the grouped split correctly removes.

**Leakage checks -- three deliberate attacks, all confirming the final feature set is clean:** (1) adding the label itself (`avg_position_win`) as a feature sent AUC from 0.561 to 0.997 -- the "confession test" proving the harness detects real leakage; (2) `gsc_clicks` showed a real, moderate gap; (3) the query-mix block showed the largest gap of all (+0.171) and was excluded as a result -- a finding that overturned my own initial assumption that it was probably safe.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata

NUM_FEATURES = ["word_count", "content_age_days", "search_volume", "competition", "backlinks"]
CAT_FEATURES = ["content_type", "main_intent"]
FEATURE_COLS = NUM_FEATURES + CAT_FEATURES

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM_FEATURES),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_FEATURES),
])

# Honest rule baseline (nonzero medians + continuous tiebreak)
def positive_median(s):
    pos = s[s > 0]
    return pos.median() if len(pos) else s.median()

med_sv, med_comp, med_bl = positive_median(data["search_volume"]), positive_median(data["competition"]), positive_median(data["backlinks"])
data["rule_score"] = ((data["search_volume"] >= med_sv).astype(int) + (data["competition"] <= med_comp).astype(int) +
                       (data["backlinks"] >= med_bl).astype(int) + (data["content_age_days"] >= 90).astype(int))
data["rule_tiebreak"] = (rankdata(data["backlinks"]) + rankdata(data["search_volume"]) + rankdata(-data["competition"])) / 3

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, data["is_page_one"], data["client_hash_id"]))
print(f"train: {len(train_idx):,} pages | test: {len(test_idx):,} pages | client overlap: "
      f"{len(set(data.iloc[train_idx]['client_hash_id']) & set(data.iloc[test_idx]['client_hash_id']))} (must be 0)")


train: 49,843 pages | test: 4,049 pages | client overlap: 0 (must be 0)


## 4. Results (vs baseline)

Same data, same grouped split, same held-out clients -- rule baseline, Logistic Regression, and Random Forest compared side by side.

In [5]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.mean(np.asarray(y_true)[order]))

X_train, X_test = data.iloc[train_idx][FEATURE_COLS], data.iloc[test_idx][FEATURE_COLS]
y_train, y_test = data.iloc[train_idx]["is_page_one"], data.iloc[test_idx]["is_page_one"]

logit = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))]).fit(X_train, y_train)
rf    = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=42, n_jobs=-1))]).fit(X_train, y_train)

logit_proba, rf_proba = logit.predict_proba(X_test)[:, 1], rf.predict_proba(X_test)[:, 1]
rule_test = data.iloc[test_idx]
rule_scores = rule_test["rule_score"] + (rule_test["rule_tiebreak"] / rule_test["rule_tiebreak"].max() * 0.99)

rows = []
for name, scores in [("Base rate (majority class)", np.full(len(y_test), y_test.mean())),
                      ("Rule baseline", rule_scores.values),
                      ("Logistic Regression", logit_proba),
                      ("Random Forest", rf_proba)]:
    rows.append({"model": name,
                 "Precision@20": round(precision_at_k(y_test.values, scores, 20), 3),
                 "Precision@50": round(precision_at_k(y_test.values, scores, 50), 3),
                 "ROC_AUC": round(roc_auc_score(y_test, scores), 3) if name != "Base rate (majority class)" else None,
                 "Avg_Precision": round(average_precision_score(y_test, scores), 3) if name != "Base rate (majority class)" else None})

comparison = pd.DataFrame(rows)
print(f"held-out base rate: {y_test.mean():.3f}")
comparison


held-out base rate: 0.536


,model,Precision@20,Precision@50,ROC_AUC,Avg_Precision
0,Base rate (majority class),0.50,0.48,NaN,NaN
1,Rule baseline,0.55,0.48,0.479,0.523
2,Logistic Regression,0.45,0.50,0.561,0.589
3,Random Forest,0.45,0.52,0.516,0.545


**Honest headline: Logistic Regression is the best-performing method on the two metrics that are stable at this sample size (ROC AUC, Average Precision).** Precision@20/@50 are noisy point estimates on only 8 held-out clients (4,049 pages) and should not be over-read in isolation. Critically, the reported numbers here are the **grouped-split (honest)** numbers -- the same model scored under a random row-level split instead showed AUC 0.654 / AP 0.656, a +0.093 / +0.068 illusion from client-memorization that would have overstated performance had the honest split not been used and directly compared (ML-09).

## 5. Limitations

1. **Single month, single portfolio.** Trained and validated on March 2026 data from one company's pseudonymized client set -- not shown to generalize to other months, seasons, or organizations.
2. **Small held-out validation.** The honest performance numbers rest on only 8 held-out clients (4,049 pages) -- a real but statistically thin validation group; Precision@K point estimates especially should be read as directional, not precise.
3. **Associational, not causal.** Every finding here is a cross-sectional, observed pattern in one snapshot. No design here supports "doing X will cause Y" -- only "pages with X are, in this data, more often page-one."
4. **Missing-data pages are less reliable.** `search_volume`, `competition`, and `backlinks` are heavily zero-inflated; roughly 43% of the model's top-tier OPPORTUNITY recommendations have all three simultaneously at zero -- very likely missing keyword/backlink data rather than genuine zero demand, and the model was found (via error analysis) to be more often confidently wrong on exactly these pages.
5. **Extreme-value sensitivity.** Logistic Regression can react strongly to extreme feature values (e.g. an unusually short page can receive a very high score primarily from the word-count coefficient alone) -- a reminder that top-ranked pages still warrant a human read before acting, not blind trust in the score.
6. **Query-mix signals are excluded, not resolved.** The `fact_content_query_90d` table showed a real leakage signal and was excluded rather than fixed -- a genuinely useful signal source is left on the table until its window-alignment against the label period can be verified precisely.

In [6]:
print("Limitations above are stated directly from findings across ML-04 through ML-10 -- no additional query needed.")


Limitations above are stated directly from findings across ML-04 through ML-10 -- no additional query needed.


## 6. Ranked recommendations

The action playbook output (ML-10) -- reading back the actual saved queue, not recomputing it, to demonstrate the pipeline is genuinely reproducible end to end.

In [7]:
playbook = pd.read_csv("work/outputs/action_playbook.csv")
tier_counts = pd.read_csv("work/outputs/tier_counts.csv", index_col=0)

print("Tier counts (from the saved w07 output):")
print(tier_counts)
print(f"\nOPPORTUNITY pages with likely-missing demand data (zero search_volume AND zero backlinks): "
      f"{((playbook['search_volume'] == 0) & (playbook['backlinks'] == 0)).sum():,} of {len(playbook):,}")

playbook.head(10)


Tier counts (from the saved w07 output):
              count
tier               
LOW_PRIORITY  26946
PROTECT       18199
OPPORTUNITY    8747

OPPORTUNITY pages with likely-missing demand data (zero search_volume AND zero backlinks): 3,730 of 8,747


,rank,content_hash_id,client_hash_id,model_proba,tier,reason_code,word_count,content_age_days,search_volume,competition,backlinks,is_page_one
0,1,content_1d10840d84866345,client_e5c2aa26a8598242,0.898374,OPPORTUNITY,strong_backlinks+real_demand,3902,54.0,30,0.95,538652,0
1,2,content_199ccff5416079a4,client_400c21c81c8b46ef,0.830773,OPPORTUNITY,strong_backlinks+real_demand+concise_content+r...,1737,0.0,90,0.00,30712,0
2,3,content_483ecb07a65bf7ab,client_2094c6eb080311d5,0.830770,OPPORTUNITY,concise_content+recently_published,109,25.0,10,0.13,0,0
3,4,content_b678de5511c085dd,client_2094c6eb080311d5,0.827498,OPPORTUNITY,strong_backlinks+real_demand+concise_content+r...,278,5.0,110,0.22,223,0
4,5,content_a95dc4a0751ba10d,client_400c21c81c8b46ef,0.812983,OPPORTUNITY,strong_backlinks+real_demand+concise_content+r...,1469,0.0,40,0.00,117938,0
5,6,content_d2ace6712a61cc0a,client_2094c6eb080311d5,0.803850,OPPORTUNITY,strong_backlinks+concise_content+recently_publ...,225,19.0,10,0.00,188,0
6,7,content_8449bd7445e204a2,client_2094c6eb080311d5,0.790906,OPPORTUNITY,concise_content,147,54.0,0,0.00,0,0
7,8,content_27a25fee9b54bb72,client_2094c6eb080311d5,0.789361,OPPORTUNITY,real_demand+concise_content+recently_published,480,12.0,590,0.01,63,0
8,9,content_c90a6416f1bf82f4,client_2094c6eb080311d5,0.777927,OPPORTUNITY,concise_content+recently_published,456,26.0,10,0.19,0,0
9,10,content_66137b3c4dac7342,client_fef1a8f436438636,0.771710,OPPORTUNITY,concise_content+recently_published,1194,0.0,10,0.00,0,0


## 7. Artifacts the paper embeds

Confirming the charts and tables the deployed page will reference actually exist on disk, generated by earlier notebooks -- not regenerated here, to prove the outputs folder is the real, working link between the notebooks and the paper.

In [8]:
import os

artifacts = [
    "work/outputs/action_playbook.csv",
    "work/outputs/tier_counts.csv",
    "work/outputs/baseline_action_score.csv",
    "work/outputs/figures/signal_importance.png",
]
for path in artifacts:
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path):,} bytes" if exists else "MISSING"
    print(f"{'OK ' if exists else 'MISSING'} {path:45} {size}")


OK  work/outputs/action_playbook.csv              1,208,509 bytes
OK  work/outputs/tier_counts.csv                  65 bytes
OK  work/outputs/baseline_action_score.csv        5,512,641 bytes
OK  work/outputs/figures/signal_importance.png    62,294 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
